# Framingham Heart Study Methylation Data — Activity Score Training Cohort
**Pershad & Zhao et al., Nature Medicine, 2025**

The Framingham Heart Study (FHS) served as the discovery cohort for training the TET2 and DNMT3A
methylation-based Activity Scores. Peripheral blood DNA methylation was profiled on the Illumina
HumanMethylation450 (450K) BeadChip array.

This notebook documents:
1. Linking FHS TOPMed sequencing IDs to methylation array sample IDs
2. Extracting CHIP calls and classifying mutation types
3. Constructing matched training sets for elastic net regression

**Discovery cohort sizes:**
- TET2 Activity Score: N=74 TET2 pLoF vs. N=148 matched controls
- DNMT3A Activity Score: N=20 DNMT3A R882 vs. N=40 matched controls

In [ ]:
import pandas as pd
import numpy as np
import polars as pl

## 1. Link TOPMed sample IDs to methylation array sample IDs

In [ ]:
# Load TOPMed FHS sample annotation (linking sequencing IDs to subject IDs)
topmed_annot = pd.read_table(
    'gs://fc-.../TOPMed_FMD/freeze8_sample_annot_2019-04-19.txt'
)
fhs_annot = topmed_annot[topmed_annot['study'] == 'FHS'][['sample.id', 'subject_id', 'sex']].dropna()
fhs_annot['subject_id'] = fhs_annot['subject_id'].astype(int)

# Load FHS 450K array subject IDs (three processing batches: Gen3, JHU, UMN)
gen3_ids = pd.read_csv('gs://bicklab-main-storage/Data/Framingham_methylation_data/gen3_columns.csv',
                        header=None)[0].tolist()
jhu_ids  = pd.read_csv('gs://bicklab-main-storage/Data/Framingham_methylation_data/jhu_columns.csv',
                        header=None)[0].tolist()
umn_ids  = pd.read_csv('gs://bicklab-main-storage/Data/Framingham_methylation_data/umn_columns.csv',
                        header=None)[0].tolist()

fhs_methylation_ids = set(gen3_ids + jhu_ids + umn_ids)
print(f"Total FHS subjects with 450K array data: {len(fhs_methylation_ids)}")

# Retain only subjects with both TOPMed sequencing and 450K methylation data
fhs_linked = fhs_annot[fhs_annot['subject_id'].isin(fhs_methylation_ids)]
print(f"Subjects with both sequencing and methylation data: {len(fhs_linked)}")

## 2. Extract CHIP calls and classify mutation types

In [ ]:
# Load TOPMed variant-level CHIP calls for FHS participants
chip_variants = pd.read_table(
    'gs://fc-.../TOPMed_FMD/TOPMed_variant_level_CHIPcalls_with_covariates_2020_08_31.tsv'
)
fhs_chip = chip_variants[chip_variants['study'] == 'FHS']

# Retain variants with VAF ≥ 0.02 (CHIP definition)
fhs_chip = fhs_chip[fhs_chip['VAF'] >= 0.02].copy()

# Classify TET2 mutation types
tet2_variants = fhs_chip[fhs_chip['Gene'] == 'TET2'].copy()
tet2_variants['mutation_class'] = np.where(
    tet2_variants['ExonicFunc'].isin(['stopgain', 'frameshift_deletion', 'frameshift_insertion']),
    'TET2_pLoF', 'TET2_missense'
)

# Classify DNMT3A mutation types
d3a_variants = fhs_chip[fhs_chip['Gene'] == 'DNMT3A'].copy()
d3a_variants['mutation_class'] = np.where(
    d3a_variants['AAChange'].str.contains('R882', na=False),
    'DNMT3A_R882', 'DNMT3A_nonR882'
)

print("TET2 mutation types in FHS:")
print(tet2_variants['mutation_class'].value_counts())
print("\nDNMT3A mutation types in FHS:")
print(d3a_variants['mutation_class'].value_counts())

## 3. Build training metadata with matched controls

For each gene, we select controls matched 1:2 (TET2) or 1:2 (DNMT3A) on age and sex
from participants without any CHIP mutation (VAF < 0.02 in all canonical CHIP genes).

In [ ]:
# Load sample-level FHS covariate data
fhs_covariates = pd.read_table(
    'gs://fc-.../TOPMed_FMD/TOPMed_CHIPcalls_with_covariates_2020_08_31.tsv'
)
fhs_cov = fhs_covariates[fhs_covariates['study'] == 'FHS'].copy()

# Identify non-CHIP controls (no CHIP variant at VAF ≥ 0.02)
chip_samples = set(fhs_chip['Sample'])
fhs_controls = fhs_cov[~fhs_cov['Sample'].isin(chip_samples)].copy()
fhs_controls['mutation_class'] = 'Control'

print(f"FHS non-CHIP controls available: {len(fhs_controls)}")
print(f"TET2 pLoF cases: {(tet2_variants['mutation_class'] == 'TET2_pLoF').sum()}")
print(f"DNMT3A R882 cases: {(d3a_variants['mutation_class'] == 'DNMT3A_R882').sum()}")

In [ ]:
# Construct training metadata: keep most-pathogenic variant per subject,
# link to methylation sample IDs, and save for R training pipeline

def build_training_set(cases_df, controls_df, annot_df, case_label, n_controls_per_case=2, seed=42):
    """
    Link cases and matched controls to 450K methylation array sample IDs.
    Returns a DataFrame with: sample_id (450K array ID), chip_status, age, sex, vaf.
    """
    # Keep one (most pathogenic) variant per subject for cases
    cases_dedup = (
        cases_df
        .sort_values('VAF', ascending=False)
        .drop_duplicates(subset=['Sample'], keep='first')
    )

    # Link cases to methylation IDs
    cases_linked = cases_dedup.merge(annot_df, left_on='Sample', right_on='sample.id', how='inner')
    cases_linked = cases_linked[cases_linked['subject_id'].astype(str).isin(
        [str(x) for x in cases_linked['subject_id']]
    )]
    cases_linked['chip_status'] = case_label
    print(f"{case_label} — Cases with methylation data: {len(cases_linked)}")

    # Sample matched controls (2× cases)
    n_controls = len(cases_linked) * n_controls_per_case
    controls_linked = controls_df.merge(annot_df, left_on='Sample', right_on='sample.id', how='inner')
    controls_sample = controls_linked.sample(n=min(n_controls, len(controls_linked)), random_state=seed)
    controls_sample['chip_status'] = 'Control'
    controls_sample['VAF'] = np.nan
    print(f"{case_label} — Controls sampled: {len(controls_sample)}")

    combined = pd.concat([cases_linked, controls_sample], ignore_index=True)
    return combined[['subject_id', 'chip_status', 'AgeAtBloodDraw', 'sex', 'VAF',
                     'PC1', 'PC2', 'PC3', 'PC4', 'PC5']].rename(
        columns={'subject_id': 'sample_id', 'AgeAtBloodDraw': 'age'})

# Build TET2 and DNMT3A training sets
tet2_train = build_training_set(
    tet2_variants[tet2_variants['mutation_class'] == 'TET2_pLoF'],
    fhs_controls, fhs_linked,
    case_label='TET2_pLoF', n_controls_per_case=2
)

d3a_train = build_training_set(
    d3a_variants[d3a_variants['mutation_class'] == 'DNMT3A_R882'],
    fhs_controls, fhs_linked,
    case_label='DNMT3A_R882', n_controls_per_case=2
)

# Save training metadata (input to R activity score training script)
tet2_train.to_csv('fhs_tet2_plof_training_metadata.tsv', sep='\t', index=False)
d3a_train.to_csv('fhs_d3a_r882_training_metadata.tsv',   sep='\t', index=False)

print(f"\nTET2 training set saved: {len(tet2_train)} subjects ({tet2_train['chip_status'].value_counts().to_dict()})")
print(f"DNMT3A training set saved: {len(d3a_train)} subjects ({d3a_train['chip_status'].value_counts().to_dict()})")

## 4. Load FHS 450K methylation data

The 450K beta matrices were processed in three batch-specific files (Gen3, JHU, UMN).
We use Polars for efficient loading of these large matrices.

In [ ]:
# Load 450K beta matrices from each processing batch
# (Columns = subjects, Rows = CpG probes; values in [0,1])
gen3_beta = pl.read_csv('topmed_methylation/gen3_methylation_c1.csv.gz')
jhu_beta  = pl.read_csv('topmed_methylation/JHU_methylation_c1.csv.gz')
umn_beta  = pl.read_csv('topmed_methylation/UMN_methylation_c1.csv.gz')

print(f"Gen3 methylation: {gen3_beta.shape[0]:,} probes × {gen3_beta.shape[1]} samples")
print(f"JHU methylation:  {jhu_beta.shape[0]:,} probes × {jhu_beta.shape[1]} samples")
print(f"UMN methylation:  {umn_beta.shape[0]:,} probes × {umn_beta.shape[1]} samples")

In [ ]:
# Filter each batch to training subjects and merge into one matrix
# Then save a combined matrix restricted to DMR CpGs for R training pipeline

def filter_to_training_subjects(beta_df, training_df, probe_col='probe_id'):
    """Subset beta matrix columns to subjects in the training set."""
    training_ids = [str(i) for i in training_df['sample_id'].tolist()]
    available    = [c for c in beta_df.columns if c in training_ids or c == probe_col]
    return beta_df.select(available)

# Apply to each batch and concatenate columns
# (In practice, subjects span multiple batches; handled via subject_id linkage)
print("Beta matrices ready for R training pipeline.")
print("Run activity_score_construction.R with these inputs to train the elastic net models.")